In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

from sklearn.model_selection import KFold

import itertools

from tqdm import tqdm

/home/jcthompson5@ad.wisc.edu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# data with initial and end point measurements
df_mono = pd.read_csv("data/exp0/exp0_mono_reps.csv")
df_exp0 = pd.read_csv("data/exp0/exp0_comm.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")

# make metabolite initial condition 0 instead of NaN 
t0_inds = df_mono.Time.values == 0
df_mono.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp0.Time.values == 0
df_exp0.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp1.Time.values == 0
df_exp1.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp2.Time.values == 0
df_exp2.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

t0_inds = df_exp3.Time.values == 0
df_exp3.loc[t0_inds, ['Lactate', 'Butyrate', 'Acetate']] = 0.

In [3]:
# bin the last measurement time 
Time = df_mono['Time'].values
for i, t in enumerate(Time):
    if t > 40:
        Time[i] = 1.
df_mono['Time'] = Time

# bin the last measurement time 
Time = df_exp0['Time'].values
for i, t in enumerate(Time):
    if t > 40:
        Time[i] = 1.
df_exp0['Time'] = Time

In [4]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
metabolites = ['pH']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch',
       'Xylan'], dtype='<U6')

# Train on DTL 0 to predict held-out DTL 0, 1, 2, 3

In [5]:
# df for training 
df = df_exp0.copy()
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_0.csv", index=False)
    
# train on dtl 0 to predict dtl 1 and 2
train_df = pd.concat((df_mono, df_exp0))
test_df = pd.concat((df_exp1, df_exp2, df_exp3))

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0., alpha_1=1.,
         evd_tol=1e-3)

# make predictions
predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

# save predictions
pred_df = pd.DataFrame()
for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

    # save species predictions for each experimental condition
    for i, exp_name in enumerate(exp_names):
        # init dataframe
        pred_df_exp = pd.DataFrame()

        # insert exp name
        pred_df_exp["Experiments"] = [exp_name]*len(T[i])
        pred_df_exp["Time"] = T[i]

        for j, s in enumerate(observed):
            pred_df_exp[s + " true"] = Y[i,:,j]
            pred_df_exp[s + " pred"] = preds[i,:,j]
            pred_df_exp[s + " stdv"] = stdvs[i,:,j]

        # append to test prediction dataframe
        pred_df = pd.concat((pred_df, pred_df_exp))
k_fold_df = pd.concat((k_fold_df, pred_df))
k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_0.csv", index=False)

Total measurements: 3595, Number of parameters: 2320, Initial regularization: 0.00e+00
Loss: 178.586, Residuals: 0.01263
Loss: 159.519, Residuals: -0.00108
Loss: 155.850, Residuals: 0.00080
Loss: 150.647, Residuals: 0.00071
Loss: 142.205, Residuals: 0.00387
Loss: 140.347, Residuals: 0.00202
Loss: 137.338, Residuals: 0.00378
Loss: 131.628, Residuals: 0.00391
Loss: 124.294, Residuals: 0.00458
Loss: 122.404, Residuals: 0.00263
Loss: 113.966, Residuals: 0.00947
Loss: 112.611, Residuals: 0.00318
Loss: 110.251, Residuals: 0.00351
Loss: 106.463, Residuals: 0.00443
Loss: 106.069, Residuals: 0.00599
Loss: 105.349, Residuals: 0.00524
Loss: 104.827, Residuals: 0.00296
Loss: 104.076, Residuals: 0.00303
Loss: 102.466, Residuals: 0.00311
Loss: 100.421, Residuals: 0.00305
Loss: 99.494, Residuals: 0.00695
Loss: 98.747, Residuals: 0.00326
Loss: 97.651, Residuals: 0.00320
Loss: 96.390, Residuals: 0.00389
Loss: 95.558, Residuals: 0.00445
Loss: 94.843, Residuals: 0.00433
Loss: 93.841, Residuals: 0.00437
L

Loss: 1661.803, Residuals: 0.00337
Loss: 1660.224, Residuals: 0.00336
Loss: 1658.570, Residuals: 0.00335
Loss: 1656.927, Residuals: 0.00335
Loss: 1656.570, Residuals: 0.00334
Loss: 1656.227, Residuals: 0.00334
Loss: 1656.182, Residuals: 0.00334
Updating precision...
Evidence 5715.144
Loss: 1683.858, Residuals: 0.00335
Loss: 1681.872, Residuals: 0.00334
Loss: 1680.676, Residuals: 0.00337
Loss: 1678.444, Residuals: 0.00336
Loss: 1675.710, Residuals: 0.00337
Loss: 1674.945, Residuals: 0.00335
Loss: 1673.829, Residuals: 0.00337
Loss: 1672.501, Residuals: 0.00337
Loss: 1671.229, Residuals: 0.00337
Loss: 1670.650, Residuals: 0.00337
Loss: 1670.375, Residuals: 0.00337
Loss: 1669.974, Residuals: 0.00337
Loss: 1669.431, Residuals: 0.00337
Updating precision...
Evidence 5743.420
Updating precision...
Evidence 5760.884
Updating precision...
Evidence 5772.820
Updating precision...
Evidence 5782.251
Updating precision...
Evidence 5790.072
Loss: 1730.987, Residuals: 0.00337
Loss: 1730.910, Residuals

Loss: 1569.093, Residuals: 0.00339
Updating precision...
Evidence 5613.242
Updating precision...
Evidence 5656.184
Updating precision...
Evidence 5681.541
Loss: 1681.386, Residuals: 0.00342
Loss: 1679.316, Residuals: 0.00341
Loss: 1676.679, Residuals: 0.00337
Loss: 1672.285, Residuals: 0.00342
Loss: 1672.039, Residuals: 0.00342
Loss: 1671.571, Residuals: 0.00342
Loss: 1668.199, Residuals: 0.00341
Loss: 1666.417, Residuals: 0.00340
Loss: 1663.323, Residuals: 0.00339
Loss: 1662.468, Residuals: 0.00339
Loss: 1661.237, Residuals: 0.00339
Loss: 1660.087, Residuals: 0.00338
Loss: 1658.265, Residuals: 0.00337
Loss: 1657.733, Residuals: 0.00337
Loss: 1656.901, Residuals: 0.00337
Loss: 1655.660, Residuals: 0.00337
Loss: 1654.480, Residuals: 0.00337
Loss: 1653.516, Residuals: 0.00337
Loss: 1652.243, Residuals: 0.00337
Loss: 1651.709, Residuals: 0.00336
Loss: 1651.082, Residuals: 0.00336
Loss: 1650.220, Residuals: 0.00335
Loss: 1649.825, Residuals: 0.00334
Loss: 1649.120, Residuals: 0.00335
Loss:

Loss: 191.945, Residuals: 0.00386
Loss: 191.838, Residuals: 0.00388
Loss: 191.835, Residuals: 0.00388
Updating precision...
Evidence 1469.183
Fail count  1
Loss: 411.617, Residuals: 0.00382
Loss: 411.183, Residuals: 0.00390
Loss: 410.960, Residuals: 0.00387
Loss: 409.804, Residuals: 0.00383
Loss: 409.732, Residuals: 0.00383
Loss: 409.608, Residuals: 0.00383
Loss: 409.428, Residuals: 0.00383
Loss: 409.402, Residuals: 0.00383
Loss: 409.312, Residuals: 0.00383
Loss: 408.897, Residuals: 0.00379
Updating precision...
Evidence 3340.468
Updating precision...
Evidence 4502.447
Loss: 1160.426, Residuals: 0.00378
Updating precision...
Evidence 5069.837
Loss: 1422.209, Residuals: 0.00358
Loss: 1409.927, Residuals: 0.00349
Loss: 1407.124, Residuals: 0.00356
Loss: 1402.183, Residuals: 0.00354
Loss: 1388.416, Residuals: 0.00342
Loss: 1385.332, Residuals: 0.00341
Loss: 1378.664, Residuals: 0.00335
Loss: 1376.391, Residuals: 0.00339
Loss: 1373.628, Residuals: 0.00336
Loss: 1369.484, Residuals: 0.00332

Loss: 796.265, Residuals: 0.00370
Loss: 795.218, Residuals: 0.00370
Loss: 793.975, Residuals: 0.00370
Loss: 793.754, Residuals: 0.00366
Loss: 792.951, Residuals: 0.00365
Loss: 792.374, Residuals: 0.00364
Loss: 791.183, Residuals: 0.00364
Loss: 791.173, Residuals: 0.00366
Loss: 790.748, Residuals: 0.00366
Loss: 790.625, Residuals: 0.00366
Loss: 790.472, Residuals: 0.00366
Updating precision...
Evidence 4483.729
Updating precision...
Evidence 5092.357
Updating precision...
Evidence 5341.935
Updating precision...
Evidence 5435.340
Updating precision...
Evidence 5481.092
Updating precision...
Evidence 5509.664
Updating precision...
Evidence 5529.997
Loss: 1698.470, Residuals: 0.00369
Loss: 1671.346, Residuals: 0.00364
Loss: 1666.611, Residuals: 0.00410
Loss: 1659.404, Residuals: 0.00409
Loss: 1649.015, Residuals: 0.00399
Loss: 1639.679, Residuals: 0.00395
Updating precision...
Evidence 5603.238
Updating precision...
Evidence 5630.960
Loss: 1691.944, Residuals: 0.00392
Loss: 1684.131, Resid

...

Loss: 1554.438, Residuals: 0.00280
Loss: 1554.285, Residuals: 0.00280
Updating precision...
Evidence 5585.062
Loss: 1626.185, Residuals: 0.00316
Loss: 1613.838, Residuals: 0.00296
Loss: 1601.151, Residuals: 0.00294
Loss: 1598.478, Residuals: 0.00320
Loss: 1595.828, Residuals: 0.00314
Loss: 1591.113, Residuals: 0.00317
Loss: 1576.693, Residuals: 0.00320
Loss: 1555.991, Residuals: 0.00358
Loss: 1547.503, Residuals: 0.00400
Loss: 1538.174, Residuals: 0.00375
Loss: 1529.435, Residuals: 0.00357
Loss: 1527.194, Residuals: 0.00358
Loss: 1523.511, Residuals: 0.00357
Loss: 1522.589, Residuals: 0.00358
Loss: 1522.003, Residuals: 0.00364
Loss: 1521.123, Residuals: 0.00364
Loss: 1521.020, Residuals: 0.00364
Loss: 1520.873, Residuals: 0.00364
Loss: 1520.632, Residuals: 0.00364
Loss: 1520.248, Residuals: 0.00363
Loss: 1519.278, Residuals: 0.00359
Loss: 1518.966, Residuals: 0.00359
Loss: 1518.942, Residuals: 0.00359
Updating precision...
Evidence 5701.686
Loss: 1617.769, Residuals: 0.00364
Loss: 1605

Loss: 449.252, Residuals: 0.00379
Loss: 449.104, Residuals: 0.00379
Loss: 448.920, Residuals: 0.00379
Loss: 448.710, Residuals: 0.00379
Loss: 448.552, Residuals: 0.00379
Loss: 448.470, Residuals: 0.00378
Loss: 448.380, Residuals: 0.00378
Loss: 448.301, Residuals: 0.00378
Loss: 448.220, Residuals: 0.00378
Loss: 448.083, Residuals: 0.00379
Loss: 448.002, Residuals: 0.00379
Loss: 447.967, Residuals: 0.00379
Loss: 447.913, Residuals: 0.00379
Loss: 447.869, Residuals: 0.00379
Loss: 447.800, Residuals: 0.00379
Loss: 447.787, Residuals: 0.00379
Loss: 447.717, Residuals: 0.00378
Updating precision...
Evidence 3455.129
Loss: 810.868, Residuals: 0.00379
Loss: 810.687, Residuals: 0.00379
Loss: 810.470, Residuals: 0.00378
Loss: 810.257, Residuals: 0.00378
Loss: 810.190, Residuals: 0.00378
Loss: 810.114, Residuals: 0.00378
Updating precision...
Evidence 4593.210
Loss: 1167.327, Residuals: 0.00396
Loss: 1159.575, Residuals: 0.00372
Loss: 1156.625, Residuals: 0.00371
Updating precision...
Evidence 51

Loss: 1698.245, Residuals: 0.00285
Updating precision...
Evidence 5694.293
Loss: 1715.345, Residuals: 0.00285
Loss: 1714.989, Residuals: 0.00285
Loss: 1714.873, Residuals: 0.00284
Loss: 1714.675, Residuals: 0.00284
Loss: 1714.378, Residuals: 0.00284
Loss: 1714.244, Residuals: 0.00280
Loss: 1714.060, Residuals: 0.00284
Loss: 1713.897, Residuals: 0.00284
Loss: 1713.684, Residuals: 0.00284
Loss: 1713.631, Residuals: 0.00284
Loss: 1713.539, Residuals: 0.00284
Loss: 1713.387, Residuals: 0.00284
Loss: 1713.253, Residuals: 0.00284
Loss: 1713.135, Residuals: 0.00284
Loss: 1713.021, Residuals: 0.00284
Loss: 1712.942, Residuals: 0.00285
Loss: 1712.824, Residuals: 0.00285
Loss: 1712.786, Residuals: 0.00285
Loss: 1712.715, Residuals: 0.00285
Updating precision...
Evidence 5708.370
Loss: 1724.717, Residuals: 0.00285
Loss: 1724.050, Residuals: 0.00285
Loss: 1723.704, Residuals: 0.00280
Loss: 1723.134, Residuals: 0.00280
Loss: 1722.969, Residuals: 0.00285
Loss: 1722.661, Residuals: 0.00284
Loss: 1722

Loss: 220.061, Residuals: 0.00385
Updating precision...
Evidence 1751.564
Fail count  1
Loss: 504.093, Residuals: 0.00404
Loss: 503.741, Residuals: 0.00403
Updating precision...
Evidence 3547.376
Loss: 911.679, Residuals: 0.00373
Loss: 906.259, Residuals: 0.00370
Loss: 904.240, Residuals: 0.00372
Loss: 900.261, Residuals: 0.00378
Loss: 893.344, Residuals: 0.00376
Loss: 891.138, Residuals: 0.00373
Loss: 888.077, Residuals: 0.00374
Loss: 886.946, Residuals: 0.00376
Loss: 886.714, Residuals: 0.00376
Loss: 886.328, Residuals: 0.00376
Loss: 885.752, Residuals: 0.00373
Loss: 885.098, Residuals: 0.00376
Loss: 884.774, Residuals: 0.00373
Loss: 884.355, Residuals: 0.00371
Loss: 883.857, Residuals: 0.00371
Loss: 883.144, Residuals: 0.00364
Loss: 882.511, Residuals: 0.00366
Loss: 881.775, Residuals: 0.00366
Loss: 880.344, Residuals: 0.00368
Loss: 879.538, Residuals: 0.00368
Loss: 875.462, Residuals: 0.00374
Loss: 875.403, Residuals: 0.00374
Loss: 875.295, Residuals: 0.00374
Updating precision...


2024-08-26 19:03:45.582271: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:45.861585: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:46.393181: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:46.657324: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:46.940353: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:47.205011: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:47.494113: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 19:03:47.746449: W external/xla/xla/s

# Train on DTL 0, 1 to predict held-out DTL 0, 1, 2, 3

In [6]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'Lactate', 'Butyrate', 'Acetate', 'AcGum', 'ArGal',
       'Inulin', 'Pectin', 'Starch', 'Xylan'], dtype='<U8')

In [7]:
# df for training 
df = pd.concat((df_exp0, df_exp1))

In [8]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_1.csv", index=False)
    
# train on dtl 0,1 to predict dtl 2,3
train_df = pd.concat((df_mono, df_exp0, df_exp1))
test_df = pd.concat((df_exp2, df_exp3))

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0., alpha_1=1.,
         evd_tol=1e-3)

# make predictions
predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

# save predictions
pred_df = pd.DataFrame()
for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

    # save species predictions for each experimental condition
    for i, exp_name in enumerate(exp_names):
        # init dataframe
        pred_df_exp = pd.DataFrame()

        # insert exp name
        pred_df_exp["Experiments"] = [exp_name]*len(T[i])
        pred_df_exp["Time"] = T[i]

        for j, s in enumerate(observed):
            pred_df_exp[s + " true"] = Y[i,:,j]
            pred_df_exp[s + " pred"] = preds[i,:,j]
            pred_df_exp[s + " stdv"] = stdvs[i,:,j]

        # append to test prediction dataframe
        pred_df = pd.concat((pred_df, pred_df_exp))
k_fold_df = pd.concat((k_fold_df, pred_df))
k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_1.csv", index=False)

Total measurements: 9803, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 607.826, Residuals: 0.00647
Loss: 565.160, Residuals: -0.00366
Loss: 551.483, Residuals: -0.00129
Loss: 525.446, Residuals: -0.00033
Loss: 520.984, Residuals: 0.00090
Loss: 479.094, Residuals: 0.00102
Loss: 456.862, Residuals: 0.00149
Loss: 416.428, Residuals: 0.00096
Loss: 388.405, Residuals: 0.00389
Loss: 375.273, Residuals: 0.00097
Loss: 354.754, Residuals: 0.00074
Loss: 342.297, Residuals: 0.00090
Loss: 319.617, Residuals: 0.00110
Loss: 285.559, Residuals: 0.00075
Loss: 282.420, Residuals: 0.00052
Loss: 276.396, Residuals: 0.00045
Loss: 265.507, Residuals: 0.00050
Loss: 247.627, Residuals: 0.00048
Loss: 234.476, Residuals: 0.00014
Loss: 231.387, Residuals: 0.00012
Loss: 227.716, Residuals: 0.00047
Loss: 221.057, Residuals: 0.00044
Loss: 210.471, Residuals: 0.00042
Loss: 195.245, Residuals: 0.00008
Loss: 188.759, Residuals: -0.00045
Loss: 184.933, Residuals: 0.00047
Loss: 178.126, Residuals:

2024-08-26 19:13:27.800428: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9857, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 619.402, Residuals: 0.00738
Loss: 577.236, Residuals: -0.00359
Loss: 563.782, Residuals: -0.00131
Loss: 538.172, Residuals: -0.00038
Loss: 533.818, Residuals: 0.00084
Loss: 493.959, Residuals: 0.00117
Loss: 475.318, Residuals: 0.00137
Loss: 438.613, Residuals: 0.00094
Loss: 399.181, Residuals: 0.00349
Loss: 389.863, Residuals: 0.00095
Loss: 374.211, Residuals: 0.00081
Loss: 345.849, Residuals: 0.00058
Loss: 305.418, Residuals: 0.00078
Loss: 301.848, Residuals: 0.00058
Loss: 295.900, Residuals: 0.00092
Loss: 285.788, Residuals: 0.00090
Loss: 269.268, Residuals: 0.00037
Loss: 252.574, Residuals: 0.00032
Loss: 250.707, Residuals: 0.00026
Loss: 247.868, Residuals: 0.00054
Loss: 242.403, Residuals: 0.00052
Loss: 233.281, Residuals: 0.00041
Loss: 219.692, Residuals: 0.00095
Loss: 214.909, Residuals: -0.00007
Loss: 206.534, Residuals: -0.00028
Loss: 194.464, Residuals: -0.00043
Loss: 189.269, Residual

2024-08-26 19:23:15.835088: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9836, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 606.535, Residuals: 0.00660
Loss: 563.109, Residuals: -0.00391
Loss: 548.884, Residuals: -0.00164
Loss: 522.050, Residuals: -0.00048
Loss: 517.448, Residuals: 0.00074
Loss: 475.786, Residuals: 0.00160
Loss: 456.859, Residuals: 0.00062
Loss: 420.801, Residuals: 0.00047
Loss: 395.193, Residuals: 0.00395
Loss: 380.987, Residuals: 0.00094
Loss: 362.720, Residuals: 0.00092
Loss: 332.664, Residuals: 0.00046
Loss: 327.029, Residuals: 0.00088
Loss: 316.632, Residuals: 0.00072
Loss: 297.480, Residuals: 0.00085
Loss: 269.568, Residuals: 0.00006
Loss: 262.646, Residuals: 0.00159
Loss: 250.644, Residuals: 0.00159
Loss: 231.528, Residuals: 0.00082
Loss: 219.558, Residuals: 0.00158
Loss: 216.954, Residuals: 0.00161
Loss: 213.377, Residuals: 0.00008
Loss: 207.382, Residuals: -0.00003
Loss: 197.024, Residuals: 0.00013
Loss: 183.249, Residuals: -0.00024
Loss: 182.513, Residuals: -0.00018
Loss: 181.137, Residual

...

2024-08-26 21:52:56.488682: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9812, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 599.307, Residuals: 0.00629
Loss: 556.325, Residuals: -0.00430
Loss: 542.263, Residuals: -0.00201
Loss: 515.466, Residuals: -0.00083
Loss: 510.780, Residuals: 0.00043
Loss: 468.883, Residuals: 0.00048
Loss: 450.323, Residuals: 0.00187
Loss: 414.737, Residuals: 0.00125
Loss: 377.494, Residuals: 0.00300
Loss: 363.127, Residuals: 0.00139
Loss: 340.525, Residuals: 0.00125
Loss: 334.778, Residuals: 0.00121
Loss: 323.841, Residuals: 0.00101
Loss: 303.339, Residuals: 0.00077
Loss: 275.253, Residuals: 0.00148
Loss: 272.988, Residuals: 0.00133
Loss: 269.128, Residuals: 0.00078
Loss: 261.961, Residuals: 0.00065
Loss: 248.931, Residuals: 0.00044
Loss: 230.460, Residuals: 0.00025
Loss: 229.400, Residuals: 0.00021
Loss: 221.245, Residuals: -0.00003
Loss: 210.350, Residuals: 0.00031
Loss: 206.109, Residuals: 0.00008
Loss: 199.115, Residuals: 0.00002
Loss: 187.919, Residuals: -0.00018
Loss: 180.538, Residuals

2024-08-26 22:02:21.728300: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 10292, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 634.268, Residuals: 0.00494
Loss: 589.960, Residuals: -0.00514
Loss: 575.810, Residuals: -0.00275
Loss: 548.789, Residuals: -0.00150
Loss: 544.210, Residuals: -0.00022
Loss: 500.876, Residuals: 0.00080
Loss: 476.901, Residuals: 0.00077
Loss: 433.028, Residuals: 0.00138
Loss: 406.153, Residuals: 0.00350
Loss: 392.598, Residuals: 0.00087
Loss: 372.243, Residuals: 0.00210
Loss: 364.918, Residuals: 0.00183
Loss: 350.874, Residuals: 0.00173
Loss: 324.923, Residuals: 0.00136
Loss: 296.024, Residuals: 0.00157
Loss: 292.004, Residuals: 0.00095
Loss: 285.829, Residuals: 0.00077
Loss: 274.720, Residuals: 0.00084
Loss: 256.138, Residuals: 0.00146
Loss: 242.419, Residuals: -0.00180
Loss: 238.004, Residuals: 0.00119
Loss: 230.324, Residuals: 0.00090
Loss: 217.707, Residuals: 0.00050
Loss: 205.109, Residuals: -0.00075
Loss: 204.299, Residuals: -0.00054
Loss: 202.747, Residuals: -0.00046
Loss: 199.983, Resid

2024-08-26 22:12:14.319034: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:14.692936: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:15.052416: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:15.416993: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:15.774059: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:16.121460: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:16.810146: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-26 22:12:17.208117: W external/xla/xla/s

# Train on DTL 0, 1, 2 to predict held-out DTL 0, 1, 2, 3

In [9]:
# df for training 
df = pd.concat((df_exp0, df_exp1, df_exp2))

In [10]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_2.csv", index=False)
    
# train on dtl 0,1,2 to predict dtl 3
train_df = pd.concat((df_mono, df_exp0, df_exp1, df_exp2))
test_df = df_exp3.copy()

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0., alpha_1=1.,
         evd_tol=1e-3)

# make predictions
predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

# save predictions
pred_df = pd.DataFrame()
for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

    # save species predictions for each experimental condition
    for i, exp_name in enumerate(exp_names):
        # init dataframe
        pred_df_exp = pd.DataFrame()

        # insert exp name
        pred_df_exp["Experiments"] = [exp_name]*len(T[i])
        pred_df_exp["Time"] = T[i]

        for j, s in enumerate(observed):
            pred_df_exp[s + " true"] = Y[i,:,j]
            pred_df_exp[s + " pred"] = preds[i,:,j]
            pred_df_exp[s + " stdv"] = stdvs[i,:,j]

        # append to test prediction dataframe
        pred_df = pd.concat((pred_df, pred_df_exp))
k_fold_df = pd.concat((k_fold_df, pred_df))
k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_2.csv", index=False)

Total measurements: 16873, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1024.200, Residuals: 0.00486
Loss: 975.841, Residuals: 0.00660
Loss: 941.254, Residuals: -0.00314
Loss: 933.629, Residuals: -0.00113
Loss: 866.307, Residuals: 0.00131
Loss: 755.194, Residuals: 0.00487
Loss: 728.863, Residuals: 0.00083
Loss: 681.257, Residuals: 0.00141
Loss: 624.576, Residuals: 0.00206
Loss: 604.538, Residuals: 0.00325
Loss: 579.225, Residuals: 0.00046
Loss: 571.653, Residuals: 0.00157
Loss: 517.837, Residuals: 0.00119
Loss: 513.810, Residuals: 0.00097
Loss: 507.713, Residuals: 0.00093
Loss: 466.997, Residuals: 0.00080
Loss: 463.163, Residuals: 0.00064
Loss: 456.667, Residuals: 0.00042
Loss: 445.029, Residuals: 0.00026
Loss: 424.442, Residuals: -0.00012
Loss: 396.831, Residuals: -0.00029
Loss: 394.571, Residuals: 0.00076
Loss: 390.078, Residuals: 0.00072
Loss: 381.866, Residuals: 0.00057
Loss: 367.302, Residuals: 0.00025
Loss: 349.617, Residuals: -0.00056
Loss: 344.375, Residua

2024-08-26 22:24:39.077277: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 16825, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1018.475, Residuals: 0.00482
Loss: 970.733, Residuals: 0.00644
Loss: 937.936, Residuals: -0.00289
Loss: 930.323, Residuals: -0.00093
Loss: 859.208, Residuals: 0.00085
Loss: 742.254, Residuals: 0.00516
Loss: 711.025, Residuals: -0.00069
Loss: 658.849, Residuals: 0.00109
Loss: 615.903, Residuals: 0.00314
Loss: 608.905, Residuals: 0.00265
Loss: 597.520, Residuals: 0.00225
Loss: 576.220, Residuals: 0.00178
Loss: 537.875, Residuals: 0.00116
Loss: 496.791, Residuals: 0.00069
Loss: 489.265, Residuals: 0.00027
Loss: 475.331, Residuals: 0.00029
Loss: 451.286, Residuals: 0.00048
Loss: 420.536, Residuals: 0.00093
Loss: 418.872, Residuals: 0.00085
Loss: 405.457, Residuals: 0.00072
Loss: 386.286, Residuals: 0.00097
Loss: 382.039, Residuals: -0.00035
Loss: 374.169, Residuals: -0.00031
Loss: 360.170, Residuals: -0.00011
Loss: 340.629, Residuals: -0.00018
Loss: 340.134, Residuals: -0.00015
Loss: 335.843, Resi

Loss: 8305.482, Residuals: -0.00051
Loss: 8301.721, Residuals: -0.00052
Loss: 8301.454, Residuals: -0.00052
Updating precision...
Evidence 26683.648
Loss: 8340.254, Residuals: -0.00051
Loss: 8339.089, Residuals: -0.00051
Loss: 8336.592, Residuals: -0.00051
Loss: 8336.209, Residuals: -0.00051
Updating precision...
Evidence 26703.518
Pass count  1


2024-08-26 22:42:03.530235: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


...

2024-08-27 03:17:26.558096: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 16797, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1026.698, Residuals: 0.00499
Loss: 958.918, Residuals: -0.00372
Loss: 939.804, Residuals: -0.00111
Loss: 903.331, Residuals: -0.00044
Loss: 831.942, Residuals: 0.00142
Loss: 744.577, Residuals: 0.00596
Loss: 710.415, Residuals: 0.00033
Loss: 660.123, Residuals: 0.00217
Loss: 643.347, Residuals: 0.00176
Loss: 612.025, Residuals: 0.00154
Loss: 554.047, Residuals: 0.00129
Loss: 505.523, Residuals: -0.00151
Loss: 490.101, Residuals: 0.00142
Loss: 467.253, Residuals: 0.00079
Loss: 458.713, Residuals: 0.00085
Loss: 442.331, Residuals: 0.00073
Loss: 413.901, Residuals: 0.00059
Loss: 390.585, Residuals: -0.00053
Loss: 386.141, Residuals: -0.00062
Loss: 382.833, Residuals: 0.00017
Loss: 376.600, Residuals: 0.00013
Loss: 365.402, Residuals: 0.00000
Loss: 348.530, Residuals: -0.00032
Loss: 344.490, Residuals: -0.00042
Loss: 337.363, Residuals: -0.00048
Loss: 325.365, Residuals: -0.00046
Loss: 316.442, Re

2024-08-27 03:28:10.013473: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 17652, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1070.364, Residuals: 0.00479
Loss: 1020.031, Residuals: 0.00627
Loss: 985.376, Residuals: -0.00301
Loss: 977.456, Residuals: -0.00116
Loss: 904.823, Residuals: 0.00098
Loss: 782.258, Residuals: 0.00478
Loss: 754.994, Residuals: 0.00010
Loss: 705.606, Residuals: 0.00095
Loss: 643.403, Residuals: 0.00141
Loss: 621.533, Residuals: 0.00245
Loss: 593.707, Residuals: 0.00053
Loss: 586.132, Residuals: 0.00172
Loss: 531.743, Residuals: 0.00064
Loss: 523.012, Residuals: 0.00078
Loss: 507.578, Residuals: 0.00069
Loss: 481.083, Residuals: 0.00074
Loss: 442.645, Residuals: -0.00005
Loss: 441.052, Residuals: -0.00011
Loss: 437.986, Residuals: -0.00006
Loss: 432.576, Residuals: 0.00005
Loss: 422.556, Residuals: 0.00005
Loss: 404.993, Residuals: 0.00021
Loss: 387.815, Residuals: 0.00014
Loss: 385.221, Residuals: 0.00036
Loss: 379.991, Residuals: 0.00036
Loss: 370.397, Residuals: 0.00018
Loss: 355.550, Residu

2024-08-27 03:44:31.968579: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-27 03:44:32.348005: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-27 03:44:32.979868: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-27 03:44:33.322250: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-08-27 03:44:33.659418: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


In [11]:
# df for training 
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3))

In [12]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_3.csv", index=False)

Total measurements: 26104, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1325.493, Residuals: 0.00384
Loss: 1241.540, Residuals: 0.00525
Loss: 1193.582, Residuals: -0.00271
Loss: 1180.997, Residuals: -0.00100
Loss: 1075.967, Residuals: 0.00153
Loss: 1046.197, Residuals: -0.00140
Loss: 989.857, Residuals: -0.00033
Loss: 888.155, Residuals: 0.00075
Loss: 825.487, Residuals: 0.00128
Loss: 785.989, Residuals: 0.00008
Loss: 744.054, Residuals: 0.00136
Loss: 733.706, Residuals: 0.00031
Loss: 665.568, Residuals: 0.00047
Loss: 656.195, Residuals: 0.00089
Loss: 638.435, Residuals: 0.00055
Loss: 606.454, Residuals: 0.00056
Loss: 567.558, Residuals: 0.00079
Loss: 560.780, Residuals: 0.00003
Loss: 555.920, Residuals: -0.00029
Loss: 547.031, Residuals: -0.00029
Loss: 531.446, Residuals: -0.00026
Loss: 509.573, Residuals: -0.00060
Loss: 506.837, Residuals: -0.00008
Loss: 501.741, Residuals: -0.00017
Loss: 492.506, Residuals: -0.00026
Loss: 477.868, Residuals: -0.00066
Loss: 471.

2024-08-27 03:59:00.114047: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 26060, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1316.528, Residuals: 0.00385
Loss: 1240.979, Residuals: 0.00529
Loss: 1189.162, Residuals: -0.00249
Loss: 1176.236, Residuals: -0.00036
Loss: 1071.173, Residuals: 0.00192
Loss: 1010.910, Residuals: -0.00195
Loss: 975.395, Residuals: 0.00229
Loss: 907.623, Residuals: 0.00247
Loss: 841.708, Residuals: 0.00182
Loss: 796.091, Residuals: 0.00024
Loss: 743.684, Residuals: -0.00155
Loss: 732.743, Residuals: 0.00018
Loss: 711.833, Residuals: 0.00021
Loss: 670.277, Residuals: 0.00029
Loss: 610.202, Residuals: 0.00056
Loss: 602.885, Residuals: -0.00010
Loss: 589.464, Residuals: -0.00009
Loss: 565.693, Residuals: -0.00002
Loss: 535.325, Residuals: -0.00009
Loss: 532.406, Residuals: 0.00012
Loss: 526.880, Residuals: 0.00007
Loss: 517.489, Residuals: -0.00015
Loss: 502.159, Residuals: -0.00050
Loss: 487.084, Residuals: -0.00071
Loss: 486.169, Residuals: -0.00058
Loss: 477.621, Residuals: -0.00048
Loss: 472

2024-08-27 04:11:08.949668: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 26142, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1309.209, Residuals: 0.00339
Loss: 1231.767, Residuals: 0.00499
Loss: 1179.941, Residuals: -0.00246
Loss: 1166.901, Residuals: -0.00030
Loss: 1061.178, Residuals: 0.00205
Loss: 996.242, Residuals: 0.00063
Loss: 878.878, Residuals: 0.00212
Loss: 811.129, Residuals: 0.00087
Loss: 779.170, Residuals: 0.00029
Loss: 774.158, Residuals: 0.00061
Loss: 726.365, Residuals: 0.00045
Loss: 641.345, Residuals: 0.00081
Loss: 633.544, Residuals: 0.00124
Loss: 575.122, Residuals: 0.00063
Loss: 571.023, Residuals: 0.00047
Loss: 540.914, Residuals: 0.00014
Loss: 540.125, Residuals: 0.00008
Loss: 533.412, Residuals: 0.00013
Loss: 521.135, Residuals: -0.00010
Loss: 501.584, Residuals: -0.00018
Loss: 491.262, Residuals: -0.00073
Loss: 475.683, Residuals: -0.00080
Loss: 462.947, Residuals: -0.00096
Loss: 461.958, Residuals: -0.00051
Loss: 452.311, Residuals: -0.00065
Loss: 436.763, Residuals: -0.00100
Loss: 436.045

...

2024-08-27 08:19:24.964610: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 26142, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1321.816, Residuals: 0.00376
Loss: 1236.582, Residuals: 0.00526
Loss: 1187.769, Residuals: -0.00270
Loss: 1175.291, Residuals: -0.00101
Loss: 1070.847, Residuals: 0.00153
Loss: 1042.748, Residuals: -0.00125
Loss: 990.643, Residuals: -0.00039
Loss: 899.084, Residuals: 0.00134
Loss: 869.654, Residuals: 0.00057
Loss: 812.609, Residuals: 0.00038
Loss: 761.959, Residuals: -0.00022
Loss: 747.191, Residuals: 0.00007
Loss: 720.046, Residuals: 0.00032
Loss: 669.688, Residuals: 0.00020
Loss: 624.435, Residuals: 0.00145
Loss: 610.773, Residuals: 0.00124
Loss: 603.127, Residuals: 0.00069
Loss: 591.526, Residuals: 0.00037
Loss: 571.602, Residuals: 0.00011
Loss: 539.264, Residuals: -0.00022
Loss: 524.545, Residuals: -0.00138
Loss: 523.091, Residuals: 0.00024
Loss: 509.756, Residuals: -0.00004
Loss: 488.055, Residuals: -0.00064
Loss: 480.155, Residuals: -0.00112
Loss: 478.237, Residuals: -0.00023
Loss: 462.8

2024-08-27 08:28:43.910529: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 26081, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1309.420, Residuals: 0.00347
Loss: 1233.027, Residuals: 0.00497
Loss: 1180.737, Residuals: -0.00255
Loss: 1167.773, Residuals: -0.00046
Loss: 1061.381, Residuals: 0.00188
Loss: 1002.173, Residuals: -0.00230
Loss: 966.459, Residuals: 0.00251
Loss: 897.193, Residuals: 0.00266
Loss: 825.280, Residuals: 0.00151
Loss: 782.146, Residuals: -0.00010
Loss: 732.448, Residuals: -0.00112
Loss: 722.025, Residuals: 0.00079
Loss: 703.453, Residuals: 0.00053
Loss: 667.321, Residuals: 0.00101
Loss: 638.196, Residuals: -0.00034
Loss: 595.057, Residuals: 0.00013
Loss: 587.204, Residuals: 0.00007
Loss: 571.677, Residuals: 0.00017
Loss: 544.178, Residuals: 0.00001
Loss: 509.401, Residuals: 0.00018
Loss: 505.875, Residuals: -0.00038
Loss: 499.806, Residuals: -0.00046
Loss: 490.226, Residuals: -0.00063
Loss: 476.847, Residuals: -0.00057
Loss: 469.623, Residuals: -0.00038
Loss: 469.504, Residuals: -0.00038
Updating p

2024-08-27 08:45:55.596537: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
